# Study 2 — Agent Journey Sankey Diagram
Tool call trajectory visualization: search → view → reviews → purchase

In [ ]:
import pandas as pd
import numpy as np
try:
    import plotly.graph_objects as go
    from plotly.subplots import make_subplots
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'plotly'])
    import plotly.graph_objects as go
    from plotly.subplots import make_subplots

import ipywidgets as widgets
from IPython.display import display, clear_output

In [ ]:
df = pd.read_csv('../results/study2/study2_all.csv')
print(f'Loaded {len(df):,} trials')

# Parse toolCallSequence into list of tool names
TOOL_MAP = {'search':'Search', 'view':'View', 'reviews':'Reviews', 'select':'Purchase'}

def parse_sequence(seq):
    if pd.isna(seq):
        return []
    steps = []
    for part in str(seq).split(' -> '):
        p = part.strip()
        if p.startswith('search'): steps.append('Search')
        elif p.startswith('view'): steps.append('View')
        elif p.startswith('reviews'): steps.append('Reviews')
        elif p.startswith('select'): steps.append('Purchase')
    return steps

df['steps'] = df['toolCallSequence'].apply(parse_sequence)
df['step_count'] = df['steps'].apply(len)

# Verify
print(f"Avg steps per trial: {df['step_count'].mean():.1f}")
print(f"Max steps: {df['step_count'].max()}")

In [ ]:
TOOL_COLORS = {
    'Search': '#378ADD',
    'View': '#1D9E75',
    'Reviews': '#D85A30',
    'Purchase': '#D4537E',
}
TOOL_COLORS_LIGHT = {
    'Search': 'rgba(55,138,221,0.15)',
    'View': 'rgba(29,158,117,0.15)',
    'Reviews': 'rgba(216,90,48,0.15)',
    'Purchase': 'rgba(212,83,126,0.15)',
}

def build_sankey(sub_df, title='', max_step=8):
    """Build a Plotly Sankey from a subset of trials."""
    # Count transitions: (step_i, tool_a) -> (step_i+1, tool_b)
    transitions = {}
    for _, row in sub_df.iterrows():
        steps = row['steps']
        hit = row.get('choseTarget', False)
        for i in range(min(len(steps)-1, max_step)):
            src = f"{i}: {steps[i]}"
            tgt = f"{i+1}: {steps[i+1]}"
            key = (src, tgt)
            if key not in transitions:
                transitions[key] = {'n': 0, 'hits': 0}
            transitions[key]['n'] += 1
            if hit:
                transitions[key]['hits'] += 1

    # Filter out tiny flows (< 0.5% of total)
    total = len(sub_df)
    min_flow = max(total * 0.005, 2)
    transitions = {k:v for k,v in transitions.items() if v['n'] >= min_flow}

    if not transitions:
        print('No transitions to show')
        return None

    # Build node list
    node_set = set()
    for (src, tgt) in transitions:
        node_set.add(src)
        node_set.add(tgt)
    nodes = sorted(node_set, key=lambda x: (int(x.split(':')[0]), x.split(':')[1]))
    node_idx = {n:i for i,n in enumerate(nodes)}

    # Node colors
    node_colors = []
    for n in nodes:
        tool = n.split(': ')[1]
        node_colors.append(TOOL_COLORS.get(tool, '#888'))

    # Link data
    sources, targets, values, link_colors, custom_data = [], [], [], [], []
    for (src, tgt), data in transitions.items():
        sources.append(node_idx[src])
        targets.append(node_idx[tgt])
        values.append(data['n'])
        tool = src.split(': ')[1]
        link_colors.append(TOOL_COLORS_LIGHT.get(tool, 'rgba(136,136,136,0.15)'))
        hit_rate = data['hits'] / data['n'] * 100 if data['n'] > 0 else 0
        custom_data.append(f"n={data['n']:,}  hit={hit_rate:.0f}%")

    fig = go.Figure(go.Sankey(
        arrangement='snap',
        node=dict(
            pad=20,
            thickness=18,
            line=dict(color='white', width=0.5),
            label=nodes,
            color=node_colors,
            hovertemplate='%{label}<br>Total: %{value:,}<extra></extra>',
        ),
        link=dict(
            source=sources,
            target=targets,
            value=values,
            color=link_colors,
            customdata=custom_data,
            hovertemplate='%{source.label} -> %{target.label}<br>%{customdata}<extra></extra>',
        ),
    ))

    fig.update_layout(
        title=dict(text=title, font=dict(size=16)),
        font=dict(size=12, family='Arial'),
        width=1000,
        height=500,
        margin=dict(l=10, r=10, t=50, b=20),
    )
    return fig

## All modes combined

In [ ]:
fig = build_sankey(df, title=f'Agent journey — all modes (n={len(df):,})')
if fig: fig.show()

## By input mode

In [ ]:
for mode in ['text_json', 'text_flat', 'html', 'screenshot']:
    sub = df[df['inputMode'] == mode]
    hit = sub['choseTarget'].mean()
    fig = build_sankey(sub, title=f'{mode}  (n={len(sub):,}, hit={hit:.1%})')
    if fig: fig.show()

## By condition (screenshot mode only — most dramatic)

In [ ]:
ss = df[df['inputMode'] == 'screenshot']
for cond in ['control', 'social_proof_a', 'scarcity', 'authority_a']:
    sub = ss[ss['condition'] == cond]
    hit = sub['choseTarget'].mean()
    fig = build_sankey(sub, title=f'screenshot x {cond}  (n={len(sub):,}, hit={hit:.1%})')
    if fig: fig.show()

## Interactive explorer

In [ ]:
mode_w = widgets.Dropdown(options=['all']+['text_json','text_flat','html','screenshot'], value='all', description='Mode:')
cond_w = widgets.Dropdown(options=['all','control','scarcity','social_proof_a','social_proof_b','urgency','authority_a','authority_b','price_anchoring'], value='all', description='Condition:')
agency_w = widgets.Dropdown(options=['all','vague','moderate','specific','cautious'], value='all', description='Funnel:')
cat_w = widgets.Dropdown(options=['all','serum','smartwatch','milk','dress'], value='all', description='Category:')
out = widgets.Output()

def update(*args):
    with out:
        clear_output(wait=True)
        sub = df.copy()
        if mode_w.value != 'all':
            sub = sub[sub['inputMode'] == mode_w.value]
        if cond_w.value != 'all':
            sub = sub[sub['condition'] == cond_w.value]
        if agency_w.value != 'all':
            sub = sub[sub['agency'] == agency_w.value]
        if cat_w.value != 'all':
            sub = sub[sub['categoryId'] == cat_w.value]
        if len(sub) == 0:
            print('No data for this combination')
            return
        hit = sub['choseTarget'].mean()
        title = f'{mode_w.value} x {cond_w.value} x {agency_w.value} x {cat_w.value}  (n={len(sub):,}, hit={hit:.1%})'
        fig = build_sankey(sub, title=title)
        if fig:
            fig.show()

mode_w.observe(update, 'value')
cond_w.observe(update, 'value')
agency_w.observe(update, 'value')
cat_w.observe(update, 'value')

display(widgets.HBox([mode_w, cond_w, agency_w, cat_w]))
display(out)
update()

## Side-by-side comparison: control vs social_proof_a (screenshot mode)

In [ ]:
from plotly.subplots import make_subplots

ss = df[df['inputMode'] == 'screenshot']

fig = make_subplots(rows=1, cols=2, specs=[[{'type':'sankey'},{'type':'sankey'}]],
                    subplot_titles=['Control (screenshot)', 'Social Proof A (screenshot)'])

for col_idx, cond in enumerate(['control', 'social_proof_a'], 1):
    sub = ss[ss['condition'] == cond]
    
    transitions = {}
    for _, row in sub.iterrows():
        steps = row['steps']
        hit = row.get('choseTarget', False)
        for i in range(min(len(steps)-1, 8)):
            src = f"{i}: {steps[i]}"
            tgt = f"{i+1}: {steps[i+1]}"
            key = (src, tgt)
            if key not in transitions:
                transitions[key] = {'n':0, 'hits':0}
            transitions[key]['n'] += 1
            if hit: transitions[key]['hits'] += 1
    
    min_flow = max(len(sub) * 0.01, 2)
    transitions = {k:v for k,v in transitions.items() if v['n'] >= min_flow}
    
    node_set = set()
    for (s,t) in transitions: node_set.add(s); node_set.add(t)
    nodes = sorted(node_set, key=lambda x: (int(x.split(':')[0]), x.split(':')[1]))
    node_idx = {n:i for i,n in enumerate(nodes)}
    node_colors = [TOOL_COLORS.get(n.split(': ')[1], '#888') for n in nodes]
    
    sources, targets, values, link_colors, cdata = [],[],[],[],[]
    for (s,t), d in transitions.items():
        sources.append(node_idx[s])
        targets.append(node_idx[t])
        values.append(d['n'])
        tool = s.split(': ')[1]
        link_colors.append(TOOL_COLORS_LIGHT.get(tool, 'rgba(136,136,136,0.15)'))
        hr = d['hits']/d['n']*100 if d['n']>0 else 0
        cdata.append(f"n={d['n']:,} hit={hr:.0f}%")
    
    fig.add_trace(go.Sankey(
        arrangement='snap',
        node=dict(pad=20, thickness=18, line=dict(color='white',width=0.5),
                  label=nodes, color=node_colors),
        link=dict(source=sources, target=targets, value=values,
                  color=link_colors, customdata=cdata,
                  hovertemplate='%{source.label} -> %{target.label}<br>%{customdata}<extra></extra>'),
    ), row=1, col=col_idx)

fig.update_layout(width=1200, height=500, margin=dict(l=10,r=10,t=50,b=20),
                  title='Screenshot mode: Control vs Social Proof A')
fig.show()